# Advanced Macroeconomics 4 Project — Marko Ikävalko

Asset-tested unemployment benefits in an Aiyagari (1994) economy,
motivated by the Orpo government housing-allowance reform (Sallinen, 2025).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from aiyagari import AiyagariModel, solve_general_equilibrium
from aiyagari.welfare import summarize_equilibrium, cev_summary, stationary_welfare, policy_diagnostics

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.float_format', '{:.6f}'.format)

## Part I — Model

Bellman equation for household with assets $a$ and employment state $z \in \{e, u\}$:

$$V(a,z)=\max_{a'\geq \underline a}\left\{u(c)+\beta \sum_{z'}\pi(z,z')V(a',z')\right\}$$

Asset-tested benefit rule:
$$B(a,u)=\max\{0,\ b-\phi_a\max(a-\bar a,0)\}$$

The baseline Aiyagari economy is the special case $\phi_a=0$.

Government budget: $\tau(rK+wN)=\sum_a B(a,u)\mu(a,u)$

## Part II — Baseline Stationary Equilibrium

In [ ]:
PARAMS = dict(
    beta=0.95, eta=2.0, alpha=0.36, delta=0.04,
    b=0.1, peu=0.0435, pue=0.5,
    amin=-0.5, amax=50.0, num_a=2000,
    phi_a=0.0, a_thresh=5.0,
)

baseline = AiyagariModel(K0=6.0, tau0=0.005, **PARAMS)

t0 = time.time()
solve_general_equilibrium(baseline, tol=1e-3, verbose=True)
print(f"Elapsed: {time.time()-t0:.1f}s")

In [ ]:
pd.DataFrame([summarize_equilibrium(baseline, 'Baseline')])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(baseline.a_grid, baseline.a_pol[:, 0], label='Employed')
ax.plot(baseline.a_grid, baseline.a_pol[:, 1], label='Unemployed')
ax.plot(baseline.a_grid, baseline.a_grid, 'k--', label='45°')
ax.set(title='Baseline policy functions', xlabel='Assets $a$', ylabel="$a'$")
ax.legend(); plt.show()

## Part III — Policy Comparative Statics

### Experiment 1: Varying the phase-out rate $\phi_a$

$\phi_a \in \{0.01, 0.025, 0.05, 0.10\}$, fixed $\bar a = 1.0$

In [ ]:
phi_a_grid = [0.01, 0.025, 0.05, 0.10]
phi_models, phi_summaries = [], []

t0 = time.time()
prev = baseline
for phi_a in phi_a_grid:
    mod = AiyagariModel(K0=prev.K, tau0=prev.tau, a_thresh=1.0, phi_a=phi_a, **{k:v for k,v in PARAMS.items() if k not in ('phi_a','a_thresh')})
    mod.v = prev.v.copy()
    solve_general_equilibrium(mod, tol=1e-3, verbose=False)
    phi_models.append(mod)
    phi_summaries.append(summarize_equilibrium(mod, f'phi_a={phi_a}'))
    prev = mod

print(f"Elapsed: {time.time()-t0:.1f}s")
phi_table = pd.DataFrame(phi_summaries)
phi_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(phi_table['regime'], phi_table['K'], marker='o')
axes[0].set(title='Aggregate capital', xlabel='$\\phi_a$')

axes[1].plot(phi_table['regime'], phi_table['tau'], marker='o')
axes[1].set(title='Tax rate $\\tau$', xlabel='$\\phi_a$')

for ax in axes: plt.setp(ax.get_xticklabels(), rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for i, mod in enumerate(phi_models):
    ax.plot(mod.a_grid, mod.a_pol[:, 1], lw=0.8, label=phi_summaries[i]['regime'])
ax.plot(baseline.a_grid, baseline.a_grid, 'k--', label='45°')
ax.set(title='Unemployed policy — varying $\\phi_a$', xlabel='$a$', ylabel="$a'$", xlim=(-0.5, 10))
ax.legend(); plt.show()

### Experiment 2: Varying the exempt threshold $\bar a$

$\bar a \in \{1, 2, 3, 4\}$, fixed $\phi_a = 0.025$

In [ ]:
a_thresh_grid = [1.0, 2.0, 3.0, 4.0]
thresh_models, thresh_summaries = [], []

t0 = time.time()
prev = baseline
for a_thresh in a_thresh_grid:
    mod = AiyagariModel(K0=prev.K, tau0=prev.tau, phi_a=0.025, a_thresh=a_thresh, **{k:v for k,v in PARAMS.items() if k not in ('phi_a','a_thresh')})
    mod.v = prev.v.copy()
    solve_general_equilibrium(mod, tol=1e-3, verbose=False)
    thresh_models.append(mod)
    thresh_summaries.append(summarize_equilibrium(mod, f'a_thresh={a_thresh}'))
    prev = mod

print(f"Elapsed: {time.time()-t0:.1f}s")
thresh_table = pd.DataFrame(thresh_summaries)
thresh_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(thresh_table['regime'], thresh_table['K'], marker='o')
axes[0].set(title='Aggregate capital', xlabel='$\\bar a$')
axes[1].plot(thresh_table['regime'], thresh_table['tau'], marker='o')
axes[1].set(title='Tax rate $\\tau$', xlabel='$\\bar a$')
for ax in axes: plt.setp(ax.get_xticklabels(), rotation=30)
plt.tight_layout(); plt.show()

## Part IV — Welfare Evaluation

In [ ]:
phi_cev_rows = [cev_summary(baseline, m, s['regime']) for m, s in zip(phi_models, phi_summaries)]
phi_cev_table = pd.DataFrame(phi_cev_rows)
phi_cev_table

In [ ]:
thresh_cev_rows = [cev_summary(baseline, m, s['regime']) for m, s in zip(thresh_models, thresh_summaries)]
thresh_cev_table = pd.DataFrame(thresh_cev_rows)
thresh_cev_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for label, tbl, ax in zip(['$\\phi_a$', '$\\bar a$'],
                           [phi_cev_table, thresh_cev_table], axes):
    ax.plot(tbl['regime'], tbl['cev_mean_pct'], marker='o', label='Mean')
    ax.plot(tbl['regime'], tbl['cev_employed_pct'], marker='s', label='Employed')
    ax.plot(tbl['regime'], tbl['cev_unemployed_pct'], marker='^', label='Unemployed')
    ax.axhline(0, ls='--', lw=0.8)
    ax.set(title=f'CEV by employment state — varying {label}',
           ylabel='CEV (% of baseline consumption)')
    ax.legend(); plt.setp(ax.get_xticklabels(), rotation=30)

plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for label, tbl, ax in zip(['$\\phi_a$', '$\\bar a$'],
                           [phi_cev_table, thresh_cev_table], axes):
    for pct, marker in [('cev_p10_pct','v'), ('cev_p50_pct','o'), ('cev_p90_pct','^')]:
        ax.plot(tbl['regime'], tbl[pct], marker=marker, label=pct[:6].replace('cev_',''))
    ax.axhline(0, ls='--', lw=0.8)
    ax.set(title=f'CEV distribution (wt. quantiles) — varying {label}',
           ylabel='CEV (%)')
    ax.legend(); plt.setp(ax.get_xticklabels(), rotation=30)

plt.tight_layout(); plt.show()

## Part V — Numerical Diagnostics

In [ ]:
all_models  = [baseline] + phi_models + thresh_models
all_labels  = ['baseline'] + [s['regime'] for s in phi_summaries] + [s['regime'] for s in thresh_summaries]

diag_rows = []
for label, mod in zip(all_labels, all_models):
    eq = summarize_equilibrium(mod, label)
    di = policy_diagnostics(mod)
    diag_rows.append({
        'regime': label,
        'mass_normalization': eq['mass_normalization'],
        'rel_market_clearing_residual': abs(eq['market_clearing_residual']) / abs(eq['K']),
        'mass_borrowing': eq['mass_borrowing'],
        'upper_grid_mass': di['upper_grid_mass'],
        'monotone_employed': di['monotone_employed'],
        'monotone_unemployed': di['monotone_unemployed'],
    })

pd.DataFrame(diag_rows)